In [ ]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

query_image_path = "../data/test_images/my_messy_500_peso_photo.jpg" 
database_dir = "../data/database/orb_database/"

# JUSTIFICACIÓN DEL PARÁMETRO RATIO_THRESH = 0.75
# Este valor define la rigurosidad del "Ratio Test" de Lowe. matemáticamente, exige que la distancia al mejor match sea al menos un 25% más corta que la distancia al segundo mejor match (ratio < 0.75).
# Es considerado un valor estándar recomendado y punto de equilibrio ideal. Más abajo en el codigo se explica la lógica detrás de estos métodos.
RATIO_THRESH = 0.75

# JUSTIFICACIÓN DEL PARÁMETRO MIN_MATCHES = 10
# Este parámetro establece el umbral mínimo de características robustas necesarias para declarar un "reconocimiento positivo" del billete.
# Se utiliza 10 y no otro valor ya que, primero que nada se necesitan al menos 4 para calcular la homografía, pero incluso 4 es un valor demasiado bajo con falsos positivos.
# Es por eso que decidimos usar 10, que es más del doble del mínimo matemático, para garantizar que el objeto detectado es genuinamente el billete y no una casualidad geométrica.
MIN_MATCHES = 10

def recognize_banknote(query_path, db_dir):
    print(f"Analizando: {os.path.basename(query_path)}")
    
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: No se pudo cargar la imagen de consulta.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    
    orb = cv2.ORB_create(nfeatures=2500)
    kp_query, des_query = orb.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No se encontraron características en la imagen.")
        return

    # BFMatcher con NORM_HAMMING para descriptores binarios (ORB)
    # Inicializamos el comparador por Fuerza Bruta. La justificación es que al ser un dataset pequeño, el costo computacional es manejable. Se podría haber utilizado FLANN que es más rápido para grandes datasets pero es menos preciso.
    # Parámetro normType=cv2.NORM_HAMMING: Obligatorio para descriptores binarios como ORB ya que ORB devuelve un vector de binarios y este calcula la distancia basada en la diferencia de bits (XOR) en lugar de distancia euclidiana como SIFT.
    # Parámetro crossCheck=False: Se desactiva para poder obtener los 2 vecinos más cercanos (k=2) en el paso KNN. Se explicará más adelante por qué se utiliza KNN.
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False) 
    category_votes = defaultdict(int)
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    if not db_files:
        print("Error: Base de datos .npy no encontrada.")
        return
    
    for db_file in db_files:
        des_db = np.load(db_file)
        if des_db is None or len(des_db) == 0:
            continue
            
        filename = os.path.basename(db_file)
        category = filename.split("_comp_")[0].replace("norm_clean_", "")
        
        # JUSTIFICACIÓN DEL MÉTODO KNN Y EL PARÁMETRO k=2
        # Se utiliza knnMatch() en lugar de match() simple o radiusMatch() porque KNN es más adaptativo. No depende de un umbral de distancia fijo, como radiusMatch() (que fallaría con 
        # cambios de iluminación), sino que garantiza encontrar los vecinos más cercanos sin importar a qué distancia absoluta se encuentren en el espacio del descriptor. Tampoco usamos el simple mathch() porque este solo devuelve el mejor match, lo que puede ser problemático con ORB debido a su naturaleza binaria y la posibilidad de matches ambiguos.
        # Además, utilizamos k=2 porque el Ratio Test de Lowe requiere comparar el mejor match con el segundo. No utilizamos 1 por lo ya explicado, ni m[as de 2 porque agrega complejidad y no aporta beneficios para Lowe.
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # Conteo de coincidencias mediante Ratio Test de Lowe
        # Aplicamos el "Ratio Test" de Lowe para filtrar falsos positivos ya que ORB puede generar matches ambiguos. Al tener 2 puntos tan distintos, signfica que el mejor match es significativamente mejor que el segundo, lo que sugiere una correspondencia más confiable.
        # Parámetro RATIO_THRESH = 0.75: Valor estándar recomendado.
        # Lógica: Si la distancia al mejor match (m_n[0]) es menor al 75% de la distancia al segundo mejor (m_n[1]), se considera una correspondencia robusta y no ambigua.
        good_matches = sum(1 for m_n in matches 
                           if len(m_n) == 2 and m_n[0].distance < RATIO_THRESH * m_n[1].distance)
                    
        if good_matches >= MIN_MATCHES:
            category_votes[category] += good_matches
            print(f"  {good_matches} coincidencias con: {category}")

    # Resultados finales
    if not category_votes:
        print("Resultado: Billete no reconocido.")
    else:
        results = sorted(category_votes.items(), key=lambda x: x[1], reverse=True)
        winner, score = results[0]
        
        print("-" * 40)
        print(f"BILLETE DETECTADO: {winner}")
        print(f"Total de coincidencias: {score}")
        print("-" * 40)
        
        if len(results) > 1:
            print(f"Segundo candidato: {results[1][0]} ({results[1][1]} matches)")

recognize_banknote(query_image_path, database_dir)

--- Analizando Imagen de Consulta: my_messy_500_peso_photo.jpg ---
  7 coincidencias con el componente 1000PesosBack.
  31 coincidencias con el componente 1000PesosFront.
  9 coincidencias con el componente 1000PesosFront.
  56 coincidencias con el componente 100PesosBack.
  15 coincidencias con el componente 100PesosBack.
  59 coincidencias con el componente 100PesosFront.
  7 coincidencias con el componente 100PesosFront.
  21 coincidencias con el componente 200PesosBack.
  10 coincidencias con el componente 200PesosBack.
  16 coincidencias con el componente 200PesosFront.
  26 coincidencias con el componente 20PesosBack.
  6 coincidencias con el componente 20PesosBack.
  28 coincidencias con el componente 20PesosFront.
  41 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroFront.
  11 coincidencias con el componente 20PesosPolimeroFront.
  17 coincidencias con el componente